# Modul 17: PyTorch-Tensoren, DataLoader und dichte Netze | Übungen

## Überblick

Sie untersuchen PyTorch-Tensoren, NumPy-Konvertierung, Broadcasting und Autograd. Danach erstellen Sie TensorDataset- und DataLoader-Objekte, definieren ein dichtes nn.Module, schreiben eine vollständige Trainings- und Auswertungsschleife und speichern den Modellzustand reproduzierbar.

**Zugehörige Vorlesungen**

- **Tensoren und Loader**
- **Dichtes PyTorch-Netz**

## Lernziele

Nach der Bearbeitung können Sie:

- PyTorch-Tensoren mit passenden Formen, Datentypen und kontrollierter NumPy-Speicherfreigabe erzeugen.
- Gradienten mit requires_grad und backward berechnen sowie Daten reproduzierbar mit DataLoadern bereitstellen.
- ein dichtes PyTorch-Modell trainieren, zwischen train und eval wechseln und state_dict sicher speichern und laden.

## Geprüfte Fähigkeiten

- Tensorformen, dtype, clone, detach, Broadcasting und Autograd
- TensorDataset, random_split, DataLoader, Batching und Shuffling
- nn.Module, BCEWithLogitsLoss, Optimierer, Trainingsschleife, Metriken und state_dict

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt einen kleinen binären Tabellendatensatz, teilt ihn reproduzierbar in Training, Validierung und Test und skaliert die Merkmale ohne Leakage. Alle PyTorch-Berechnungen laufen standardmäßig auf der CPU.

In [ ]:
import copy
import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

geraet = torch.device("cpu")
print("PyTorch-Version:", torch.__version__)
print("Verwendetes Gerät:", geraet)

daten = load_breast_cancer()
X_gesamt = daten.data.astype("float32")
y_gesamt = daten.target.astype("float32")

X_train_roh, X_test_roh, y_train_np, y_test_np = train_test_split(
    X_gesamt,
    y_gesamt,
    test_size=0.20,
    stratify=y_gesamt,
    random_state=RANDOM_SEED,
)
X_train_roh, X_val_roh, y_train_np, y_val_np = train_test_split(
    X_train_roh,
    y_train_np,
    test_size=0.20,
    stratify=y_train_np,
    random_state=RANDOM_SEED,
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_roh).astype("float32")
X_val_np = scaler.transform(X_val_roh).astype("float32")
X_test_np = scaler.transform(X_test_roh).astype("float32")

print("Train, Validierung, Test:", X_train_np.shape, X_val_np.shape, X_test_np.shape)

### Aufgabe 1: Tensoren, Datentypen und NumPy-Speicher kontrollieren

Erzeugen Sie aus dem vorgegebenen NumPy-Array einmal einen Tensor mit `torch.from_numpy` und einmal mit `torch.tensor`. Ändern Sie anschließend einen Wert im NumPy-Array und prüfen Sie beide Tensoren.

Erklären Sie den Unterschied zwischen gemeinsam genutztem Speicher und einer Kopie. Demonstrieren Sie außerdem `clone`, `detach` und die Rückkonvertierung eines CPU-Tensors nach NumPy. Geben Sie Form, Rang und Datentyp aus.

In [ ]:
array_basis = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Wann ist geteilter Speicher nützlich und wann riskant?

### Aufgabe 2: Indexieren, Broadcasting und Autograd prüfen

Erzeugen Sie eine Variable `x` der Form `(3, 2)` mit `requires_grad=True` und einen Biasvektor `b` der Form `(2,)`, ebenfalls mit Gradienten. Berechnen Sie

`verlust = sum((x + b)^2)`

und rufen Sie `backward()` auf. Geben Sie die Gradientenformen und -werte aus. Leiten Sie den Biasgradienten manuell her und prüfen Sie ihn mit `torch.allclose`. Setzen Sie danach die Gradienten mit `zero_()` zurück.

In [ ]:
x_autograd = torch.tensor(
    [[1.0, -1.0], [2.0, 0.5], [-0.5, 3.0]],
    dtype=torch.float32,
    requires_grad=True,
)
b_autograd = torch.tensor([0.2, -0.3], dtype=torch.float32, requires_grad=True)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum summiert sich der Biasgradient über die Batchdimension?

### Aufgabe 3: TensorDataset und DataLoader reproduzierbar erstellen

Wandeln Sie die drei Datensplits in Float32-Tensoren um. Die Merkmalsform soll `(N, 30)` und die Zielform `(N, 1)` sein. Erstellen Sie TensorDataset-Objekte und DataLoader mit Batchgröße 32.

Nur der Trainingsloader soll mischen. Verwenden Sie für das Training einen `torch.Generator` mit festem Seed. Inspizieren Sie den ersten Batch und prüfen Sie, dass Validierung und Test jede Beobachtung genau einmal enthalten.

In [ ]:
# Erzeugen Sie aus jedem Split ein TensorDataset und anschließend einen DataLoader.

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum wird nur der Trainingsloader gemischt?

### Aufgabe 4: Ein dichtes nn.Module mit passenden Logits definieren

Definieren Sie eine Klasse `DichtesNetz`, die von `nn.Module` erbt. Verwenden Sie zwei verborgene Linear-Schichten mit höchstens 32 und 16 Einheiten, ReLU und optional Dropout. Die letzte Schicht soll genau einen Logit pro Beispiel ausgeben.

Prüfen Sie die Ausgabeform eines Batches. Berechnen Sie `BCEWithLogitsLoss` und erklären Sie, warum im Modell keine Sigmoid-Schicht benötigt wird.

In [ ]:
class DichtesNetz(nn.Module):
    def __init__(self, eingabe_merkmale, dropout_rate=0.10):
        super().__init__()
        pass

    def forward(self, x):
        pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum kombiniert BCEWithLogitsLoss Sigmoid und Kreuzentropie intern?

### Aufgabe 5: Trainings-, Validierungs- und Testschleifen schreiben

Schreiben Sie Funktionen für eine Trainingsepoche und für die Auswertung ohne Gradienten. Verwenden Sie Adam, Lernrate 0.001, `BCEWithLogitsLoss` und höchstens 35 Epochen. Speichern Sie Verlust, Accuracy und F1 für Training und Validierung.

Nutzen Sie Early Stopping mit Geduld 6 und speichern Sie im Arbeitsspeicher eine Kopie des besten `state_dict`. Stellen Sie diesen Zustand wieder her und bewerten Sie das Modell einmalig auf dem Testloader. Zeichnen Sie die Verlustkurven.

In [ ]:
def trainiere_epoche(modell, loader, optimizer, loss_fn, device):
    pass

def bewerte_modell(modell, loader, loss_fn, device):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Wirkung haben model.train(), model.eval() und torch.no_grad()?

### Aufgabe 6: Integrationsaufgabe: state_dict speichern, laden und prüfen

Speichern Sie den besten Modellzustand zusammen mit wesentlichen Metadaten in einem temporären Verzeichnis. Erzeugen Sie eine neue Modellinstanz, laden Sie den Zustand mit `map_location="cpu"` und setzen Sie das Modell in den Evaluationsmodus.

Prüfen Sie, ob die ersten zehn Logits und Wahrscheinlichkeiten vor und nach dem Laden übereinstimmen. Implementieren Sie außerdem eine Funktion `sichere_einzelvorhersage`, die Form, endliche Werte und Merkmalsanzahl prüft, die gespeicherte Skalierung anwendet und Wahrscheinlichkeit sowie Klasse zurückgibt.

In [ ]:
def sichere_einzelvorhersage(rohzeile, modell, scaler_objekt, erwartete_merkmale, schwelle=0.5):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum reicht eine state_dict-Datei allein für einen späteren Einsatz nicht aus?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?